# Evaluating RAG with DeepEval — A Beginner's Guide

This notebook teaches you how to **measure the quality of a RAG (Retrieval-Augmented Generation) system** using [DeepEval](https://github.com/confident-ai/deepeval).

**Stack (all pinned to the versions you asked for):**

| Library | Version | Role |
|---|---|---|
| `deepeval` | 4.0.5 | The evaluation framework (metrics, test cases, runner) |
| `langchain` | 1.3.1 | Core LangChain |
| `langchain-groq` | 1.1.2 | Groq-hosted LLMs (fast, free tier) |
| `langchain-openai` | 1.2.2 | OpenAI LLMs |

You only need **one** LLM provider (Groq *or* OpenAI). We'll let you toggle between them.

> All code patterns here were pulled from the **latest DeepEval docs via Context7**, so the API usage is current.

**By the end you'll understand:**
1. What "evaluating RAG" actually means and why eyeballing isn't enough
2. The 5 core RAG metrics and what each one tells you
3. The 4 pieces of data every test case needs
4. How to run metrics one-by-one and read their explanations
5. How to evaluate a whole dataset at once and interpret the results


## 1. What does "evaluating RAG" mean?

A RAG system answers a question in **two steps**:

1. **Retrieve** — search a knowledge base and pull back the most relevant chunks of text (the *context*).
2. **Generate** — feed that context + the question to an LLM, which writes the final answer.

Two steps means **two places things can go wrong**:

- The *retriever* fetches the wrong or incomplete context → the LLM never had a chance.
- The *generator* gets good context but still hallucinates, ignores it, or rambles off-topic.

"Evaluating RAG" means **scoring each step separately** so you know *which* part to fix. A single thumbs-up/thumbs-down can't tell you that.

**Why not just read the answers yourself?** It doesn't scale (you can't read 1,000 answers per deploy), it's subjective, and it can't tell retrieval problems from generation problems. We need objective, repeatable numbers.


## 2. The core metrics (and what each one is really asking)

DeepEval scores RAG with **LLM-as-a-judge** metrics: another LLM reads the question, the answer, and the context, and rates them on a **0.0 → 1.0** scale (higher = better). Each metric answers one plain-English question.

### Generator metrics — *is the answer any good?*

| Metric | The question it answers | Needs |
|---|---|---|
| **Answer Relevancy** | Does the answer actually address the question (no waffle)? | question + answer |
| **Faithfulness** | Is the answer *backed by* the retrieved context, or did the LLM make things up? | answer + context |

### Retriever metrics — *did we fetch the right context?*

| Metric | The question it answers | Needs |
|---|---|---|
| **Contextual Relevancy** | Of the retrieved chunks, how much is actually relevant (vs. noise)? | question + context |
| **Contextual Recall** | Did we retrieve *everything* needed to produce the ideal answer? | context + ideal answer |
| **Contextual Precision** | Are the most relevant chunks ranked *near the top*? | question + context + ideal answer |

### The "RAG Triad"
A popular quick-start trio that covers the whole pipeline with just 3 metrics: **Answer Relevancy + Faithfulness + Contextual Relevancy**. Start here if you want minimal setup.

**Rule of thumb:**
- Faithfulness low → your generator is hallucinating.
- Contextual Recall low → your retriever is *missing* needed info (bad chunking / embeddings / search).
- Contextual Precision low → the right info is there but *buried* below junk (re-ranking would help).


## 3. Anatomy of a test case — the 4 fields

Every example you evaluate is an `LLMTestCase`. It can hold up to four fields. **Which metrics you can run depends on which fields you fill in.**

| Field | Plain meaning | Where it comes from | Used by |
|---|---|---|---|
| `input` | The user's question | Your test question | Answer Relevancy, Contextual Relevancy/Precision |
| `actual_output` | What *your* RAG actually answered | Your RAG pipeline (live) | All generator metrics |
| `retrieval_context` | The chunks your retriever returned | Your retriever (live) | All metrics |
| `expected_output` | The *ideal* answer you'd want (a.k.a. ground truth) | You write it once, by hand | Contextual Recall & Precision |

Key idea: `actual_output` and `retrieval_context` come **from running your real system**. `expected_output` is the human-written gold standard you compare against. You only need `expected_output` for the two retriever metrics that judge completeness/ranking.


## 4. Setup

Install the pinned versions. (If a pin is unavailable on PyPI, drop the `==...` for that package.)


### API keys & notebook setup

- DeepEval's metrics call an LLM, so you need an API key for **whichever provider you pick**.
  - Groq: free key from <https://console.groq.com>
  - OpenAI: key from <https://platform.openai.com>
- `nest_asyncio` lets DeepEval's async engine run inside Jupyter (avoids "event loop already running" errors).
- We opt out of telemetry so everything stays local. (Optional: run `deepeval login` in a terminal to view results on the Confident AI dashboard instead of just the console.)


In [1]:
import os
import getpass
import nest_asyncio

# Lets DeepEval's async evaluation run inside a notebook.
nest_asyncio.apply()

# Keep evaluation fully local (no data sent to Confident AI).
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

# --- Choose ONE provider ---
PROVIDER = "openai"   # "groq" or "openai"

if PROVIDER == "groq":
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")
elif PROVIDER == "openai":
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OPENAI_API_KEY: ")
else:
    raise ValueError("PROVIDER must be 'groq' or 'openai'")

print(f"Provider set to: {PROVIDER}")

Provider set to: openai


## 5. Choosing the "judge" LLM

DeepEval metrics don't compute scores with math formulas — they ask an LLM to read the texts and judge them (**LLM-as-a-judge**). So we need to give DeepEval a model to use as the judge.

A few practical notes:
- Use a **capable** model as the judge. The judge needs to reason well; a weak judge gives noisy scores. On Groq, `llama-3.3-70b-versatile` is a solid free option; on OpenAI, `gpt-4o-mini` is cheap and reliable.
- The judge should **not** be the same tiny model you're testing — keep judging separate from generating.
- DeepEval needs the judge to return **structured JSON**. The cleanest way to guarantee that is to wrap a LangChain chat model and use its `.with_structured_output(schema)`, which forces valid output and avoids parsing errors (this is the #1 source of flaky custom-model evals).

First, build the LangChain chat model:


In [2]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

if PROVIDER == "groq":
    judge_chat_model = ChatGroq(
        model="llama-3.3-70b-versatile",  # check console.groq.com for current model names
        temperature=0,                    # 0 = consistent, repeatable judging
    )
else:
    judge_chat_model = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
    )

# Quick smoke test that the model + key work:
print(judge_chat_model.invoke("Reply with exactly: OK").content)

OK


### Wrap the chat model as a DeepEval judge

DeepEval expects a class that inherits from `DeepEvalBaseLLM` and implements four methods:

- `load_model()` — return the underlying model
- `generate(prompt, schema)` — sync call, return a structured object
- `a_generate(prompt, schema)` — async version (DeepEval runs metrics concurrently)
- `get_model_name()` — a label for reports

The `schema` argument is the trick: when DeepEval passes a Pydantic schema, we use `with_structured_output(schema)` so the model is *forced* to return exactly the shape DeepEval needs. We keep it optional so the wrapper also works for plain text calls.


In [3]:
from typing import Optional
from pydantic import BaseModel
from deepeval.models.base_model import DeepEvalBaseLLM


class LangChainJudge(DeepEvalBaseLLM):
    """Wraps any LangChain chat model so DeepEval can use it as a judge."""

    def __init__(self, model, name: str):
        self.model = model
        self.name = name

    def load_model(self):
        return self.model

    def generate(self, prompt: str, schema: Optional[BaseModel] = None):
        model = self.load_model()
        if schema is not None:
            # Force valid, structured JSON output -> reliable scoring.
            return model.with_structured_output(schema).invoke(prompt)
        return model.invoke(prompt).content

    async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None):
        model = self.load_model()
        if schema is not None:
            return await model.with_structured_output(schema).ainvoke(prompt)
        res = await model.ainvoke(prompt)
        return res.content

    def get_model_name(self) -> str:
        return self.name


judge = LangChainJudge(judge_chat_model, name=f"{PROVIDER}-judge")
print("Judge ready:", judge.get_model_name())

Judge ready: openai-judge


> **OpenAI shortcut:** If you use OpenAI, you can actually skip the wrapper entirely — DeepEval supports OpenAI natively. Just set `OPENAI_API_KEY` and pass a string to any metric, e.g. `FaithfulnessMetric(model="gpt-4o-mini")`. The wrapper above is what you need for Groq (and any non-OpenAI provider), so we'll use `judge` everywhere to keep one consistent path.


## 6. A tiny RAG pipeline (so the test-case fields aren't magic)

To make `retrieval_context` and `actual_output` concrete, here's a *minimal* RAG: a 4-document knowledge base, a dead-simple keyword retriever, and the LLM as the generator.

In a real project you'd swap the retriever for your vector store (Chroma, FAISS, etc.) — but the **evaluation** code afterwards is identical. The point is just to see where each test-case field comes from.


In [4]:
# 1) A tiny knowledge base (each string = one "chunk")
KNOWLEDGE_BASE = [
    "All customers are eligible for a 30-day full refund at no extra cost.",
    "Returns must be initiated through the online portal under 'My Orders'.",
    "Our stores are open from 9am to 9pm on weekdays.",
    "Gift cards are non-refundable and cannot be exchanged for cash.",
]

# 2) A naive retriever: rank chunks by word overlap with the question.
def retrieve(question: str, k: int = 2):
    q_words = set(question.lower().split())
    scored = [
        (len(q_words & set(doc.lower().split())), doc)
        for doc in KNOWLEDGE_BASE
    ]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored[:k]]

# 3) The generator: answer ONLY from the retrieved context.
def generate_answer(question: str, context: list[str]) -> str:
    context_block = "\n".join(f"- {c}" for c in context)
    prompt = (
        "Answer the question using ONLY the context below. "
        "Be concise.\n\n"
        f"Context:\n{context_block}\n\nQuestion: {question}\nAnswer:"
    )
    # Reuse the same chat model just for the demo generation.
    return judge_chat_model.invoke(prompt).content

# Run the pipeline on one question:
question = "What if these shoes don't fit?"
retrieved = retrieve(question)
answer = generate_answer(question, retrieved)

print("QUESTION:        ", question)
print("RETRIEVED CONTEXT:", retrieved)
print("GENERATED ANSWER:", answer)

QUESTION:         What if these shoes don't fit?
RETRIEVED CONTEXT: ['All customers are eligible for a 30-day full refund at no extra cost.', "Returns must be initiated through the online portal under 'My Orders'."]
GENERATED ANSWER: You can return them for a full refund within 30 days by initiating the return through the online portal under 'My Orders'.


## 7. Build an `LLMTestCase`

Now we package the pipeline outputs into a test case. Note how each field maps to Section 3:
- `input` ← the question
- `actual_output` ← what our RAG generated
- `retrieval_context` ← the chunks our retriever returned
- `expected_output` ← the ideal answer **we** write by hand (the gold standard)


In [5]:
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input=question,
    actual_output=answer,
    retrieval_context=retrieved,
    expected_output="You can get a full 30-day refund at no extra cost.",  # gold answer
)

print("Test case built.")

Test case built.


## 8. Run metrics one at a time (and read the *reason*)

The best way to learn is to run each metric alone with `include_reason=True`. Then:
- `metric.measure(test_case)` computes the score, and
- `metric.score` (0–1) and `metric.reason` (plain-English explanation) tell you *why*.

`threshold=0.7` means "0.7 or above = pass". Tune it later to your quality bar.


In [6]:
from deepeval.metrics import FaithfulnessMetric

faithfulness = FaithfulnessMetric(model=judge, threshold=0.7, include_reason=True)
faithfulness.measure(test_case)

print(f"Faithfulness score: {faithfulness.score:.2f}")
print("Reason:", faithfulness.reason)

Output()

Faithfulness score: 1.00
Reason: The score is 1.00 because there are no contradictions present, indicating that the actual output aligns perfectly with the retrieval context.


In [7]:
from deepeval.metrics import AnswerRelevancyMetric

answer_relevancy = AnswerRelevancyMetric(model=judge, threshold=0.7, include_reason=True)
answer_relevancy.measure(test_case)

print(f"Answer Relevancy score: {answer_relevancy.score:.2f}")
print("Reason:", answer_relevancy.reason)

Output()

Answer Relevancy score: 1.00
Reason: The score is 1.00 because the response directly addresses the concern about shoe fit without any irrelevant statements.


In [8]:
from deepeval.metrics import ContextualRelevancyMetric

ctx_relevancy = ContextualRelevancyMetric(model=judge, threshold=0.7, include_reason=True)
ctx_relevancy.measure(test_case)

print(f"Contextual Relevancy score: {ctx_relevancy.score:.2f}")
print("Reason:", ctx_relevancy.reason)

Output()

Contextual Relevancy score: 0.00
Reason: The score is 0.00 because there are no relevant statements in the retrieval context that address the concern of shoe fitting, as all provided reasons highlight the focus on refund policies and return processes, which do not relate to the input question.


In [9]:
from deepeval.metrics import ContextualRecallMetric, ContextualPrecisionMetric

# These two REQUIRE expected_output (the gold answer).
ctx_recall = ContextualRecallMetric(model=judge, threshold=0.7, include_reason=True)
ctx_recall.measure(test_case)
print(f"Contextual Recall score:    {ctx_recall.score:.2f}")
print("Reason:", ctx_recall.reason)
print("-" * 60)

ctx_precision = ContextualPrecisionMetric(model=judge, threshold=0.7, include_reason=True)
ctx_precision.measure(test_case)
print(f"Contextual Precision score: {ctx_precision.score:.2f}")
print("Reason:", ctx_precision.reason)

Output()

Output()

Contextual Precision score: 1.00
Reason: The score is 1.00 because the relevant node, which addresses the concern about the shoes not fitting, is ranked first and provides a clear solution regarding the refund policy. The irrelevant node, ranked second, offers procedural information that does not directly relate to the fit issue, thus justifying its lower rank.


**How to read these:** Don't fixate on the exact decimal — the **reason** is where the value is. A low Faithfulness reason will name the specific claim that wasn't supported by context; a low Recall reason will name the missing fact. That's what tells you *what to fix*.


## 9. Evaluate a whole dataset at once

In practice you don't run metrics one case at a time — you collect many test cases into an `EvaluationDataset` and call `evaluate()` once. It runs every metric on every case (concurrently) and prints a pass/fail table.

Below we build a few cases by running our RAG over several questions, then evaluate them all together.


In [10]:
from deepeval.dataset import EvaluationDataset

questions_and_gold = [
    ("What if these shoes don't fit?",
     "You can get a full 30-day refund at no extra cost."),
    ("How do I start a return?",
     "Start a return through the online portal under 'My Orders'."),
    ("Can I get cash back for a gift card?",
     "No, gift cards are non-refundable and can't be exchanged for cash."),
]

test_cases = []
for q, gold in questions_and_gold:
    ctx = retrieve(q)
    out = generate_answer(q, ctx)
    test_cases.append(
        LLMTestCase(
            input=q,
            actual_output=out,
            retrieval_context=ctx,
            expected_output=gold,
        )
    )

dataset = EvaluationDataset()
for tc in test_cases:
    dataset.add_test_case(tc)

print(f"Dataset built with {len(dataset.test_cases)} test cases.")

Dataset built with 3 test cases.


In [11]:
from deepeval import evaluate

results = evaluate(
    test_cases=dataset.test_cases,
    metrics=[
        AnswerRelevancyMetric(model=judge, threshold=0.7),
        FaithfulnessMetric(model=judge, threshold=0.7),
        ContextualRelevancyMetric(model=judge, threshold=0.7),
        ContextualRecallMetric(model=judge, threshold=0.7),
        ContextualPrecisionMetric(model=judge, threshold=0.7),
    ],
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai-judge, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using openai-judge, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using openai-judge, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using openai-judge, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using openai-judge, strict=False, 
async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              What if these shoes don't fit?                                                       │
│  │     Actual Output:      You can return them for a full refund within 30 days by initiating the return        │
│  │                         through the online portal under 'My Orders'.                                         │
│  │     Expected Output:    You can get a full 30-day refund at no extra cost.                                   │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.70      │ The score is 1.00 because the response directly...    │
│        PASS  │ Faithfulness         │ 1.00  │ 0.70      │ The score is 1.00 because there are no contradi...    │
│        FAIL  │ Contextual Relevancy │ 0.00  │ 0.70      │ The score is 0.00 because there are no relevant       │
│              │                      │       │           │ statements in the retrieval context that address      │
│              │                      │       │           │ the concern of shoe fitting, as all provided          │
│              │                      │       │           │ reasons indicate a focus on refund policies and       │
│              │                      │       │           │ return processes, which do not relate to the input    │
│              │                      │       │           │ question.                                             │
│        PASS  │ Contextual Recall    │ 1.00  │ 0.70      │ The score is 1.00 because the sentence directly...    │
│        PASS  │ Contextual Precision │ 1.00  │ 0.70      │ The score is 1.00 because the relevant node, wh...    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              How do I start a return?                                                             │
│  │     Actual Output:      Initiate a return through the online portal under 'My Orders'.                       │
│  │     Expected Output:    Start a return through the online portal under 'My Orders'.                          │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.70      │

⚠ WARNING: No hyperparameters logged.
» ]8;id=12104292;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.76s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

`evaluate()` prints a per-case breakdown (each metric's score, pass/fail, and reason) plus an overall summary. The returned `results` object holds the same data programmatically if you want to log it to MLflow, fail a CI build, or chart trends over time.


## 10. Interpreting results & where to go next

**Reading the scoreboard — diagnose by metric:**

| Symptom | Likely culprit | What to try |
|---|---|---|
| Low **Faithfulness** | Generator hallucinating | Stronger generator; stricter "answer only from context" prompt |
| Low **Answer Relevancy** | Generator rambling / off-topic | Tighter prompt; lower temperature |
| Low **Contextual Relevancy** | Retriever returns noise | Better chunking; smaller `k`; metadata filters |
| Low **Contextual Recall** | Retriever *misses* needed info | Better embeddings; larger `k`; hybrid (BM25 + vector) search |
| Low **Contextual Precision** | Right info buried low in results | Add a re-ranker (cross-encoder) |

**Practical tips:**
- **Thresholds are yours to set.** 0.7 is a common starting bar; raise it as your system matures.
- **Cost & speed:** each metric = one+ LLM call per case. For big datasets, prefer a cheap-but-capable judge and let DeepEval run cases concurrently (it does by default).
- **Consistency:** keep judge `temperature=0` so scores are repeatable across runs.
- **Make it a regression test:** wrap `evaluate()` (or `assert_test`) in `pytest` and run it in CI, so a retrieval/prompt change that drops scores fails the build.
- **RAGAS:** a popular alternative/companion framework with overlapping metrics — worth knowing, but the concepts you learned here transfer directly.

**You now know the full loop:** build test cases from your real RAG → score retriever and generator separately → read the reasons → fix the weak component → re-run.
